## CNN model comparison 

This script implements three convolutional neural network (CNN) models for classifying muscle contractions. Using the DataFrame generated by `emg_feature_extraction.ipynb`, you can:

* Train the models
* Save the trained models
* Visualize and plot the training results

### Available Models

#### Single-Head Count

Predicts the number of muscle contractions within a given epoch.

#### Single-Head Duration

Predicts the duration of the complete contraction period within a given epoch.

#### Two-Head Model

Simultaneously predicts:

* The number of muscle contractions within a given epoch.
* The duration of the contraction period within the same epoch.


## Library Importations 

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, accuracy_score, f1_score
from matplotlib import pyplot as plt

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.regularizers import l2

## Loading in Data

### Loading in data

In [26]:
# loading in dataframe with pre-processed EMG data and extracted features from emg_feature_extraction.ipynb

features_all = pd.read_pickle("training_features_18042026.pkl")
features_all = features_all.rename(columns={'Nap Number': 'Nap_ID'}) # making column names match 
trial_info = pd.read_csv('trial_info_duration_1205.csv') # updated trial info sheet that includes contraction duration 

# add duration information for regression model 
features_all = features_all.merge(
    trial_info[['Subject', 'Nap_ID', 'Triggers_Order_Nap', 'Duration_1', 'Duration_2']],
    on=['Subject', 'Nap_ID', 'Triggers_Order_Nap'],
    how='left'
)
features_all = features_all.rename(columns={
    'Duration_1': 'Duration_corr',
    'Duration_2': 'Duration_zygo'
})


# Dropping epochs for RL03JG Nap 5 because subject info not in MATLAB dataframe  
x = np.isnan(features_all['Duration_zygo']) 
indices = np.where(x)[0]
features_all = features_all.drop(indices)

### Global Variables 

In [27]:
# initializing variables for training 
epoch_num = 10 # number of training epochs 

# initializing variables for displaying results 
metrics = ["f1", "accuracy", "mse"]
targets = ["zygo", "corr", "overall"]
splits = ["training", "testing"]
metric_tuples = [("model_name", "", "")]
metric_tuples += [ (metric, target, split)
    for metric in metrics
    for target in targets
    for split in splits]
metric_tuples += [("k-fold", "mean", ""), ("k-fold", "std", "")]

results_columns = pd.MultiIndex.from_tuples(
    metric_tuples,
    names=["metric", "target", "split"]
)
results_columns = pd.MultiIndex.from_tuples(metric_tuples)
results_df = pd.DataFrame(columns=results_columns)

### Defining training data 

In [28]:
muscle_groups = features_all["True_Muscle_Activated"].repeat(2).reset_index(drop=True)

X_zygo = features_all["Zygo"]
X_corr = features_all["Corr"]

X_zygo = np.array(X_zygo.tolist())
X_corr = np.array(X_corr.tolist())

# extract contractions 
y_zygo_contr = features_all["Num_Contractions_Zygo"].astype(int).to_numpy()
y_corr_contr = features_all["Num_Contractions_Corr"].astype(int).to_numpy()

# extract contraction duration period
y_zygo_dur = features_all["Duration_zygo"].astype(int).to_numpy()
y_corr_dur = features_all["Duration_corr"].astype(int).to_numpy()

X = np.concatenate((X_zygo, X_corr), axis=0)
y_contractions = np.concatenate((y_zygo_contr,y_corr_contr),axis=0)       
y_durations = np.concatenate((y_zygo_dur,y_corr_dur),axis=0)       

indices_1d = np.arange(len(X))
idx_train, idx_test, muscle_train, muscle_test = train_test_split(indices_1d, muscle_groups,test_size=0.2, random_state=42) 


# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]

y_train_full_contr = y_contractions[idx_train]
y_test_contr = y_contractions[idx_test]

y_train_full_dur= y_durations[idx_train]
y_test_dur = y_durations[idx_test]

y = np.column_stack((y_contractions, y_durations))
y_test = y[idx_test]
y_train_full = y[idx_train]


# training variables 
input_shape = X_train_full.shape[1:]   
input_shape = (input_shape[0], 1)   
epoch_len = np.shape(X_zygo)[1]

## Model Definitions 

### Single-head count 

In [29]:
# single head CNN model for number of contractions  
def CNN_model_contraction(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add another max pooling layer

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(num_classes, activation='softmax'))  # Add the output layer with softmax activation


    return model  # Return the compiled model

### Single-head duration 

In [30]:
# single head CNN model for duration 
def CNN_model_duration(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=3))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add another max pooling layer

    model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers

    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(1, activation='linear'))

    return model  # Return the compiled model

### Two-head for count and duration 

In [31]:
def CNN_model_twohead(input_shape, num_classes,feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    # Convolutional Layers - 
    x = Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape)(inputs) # Add a 1D convolutional layer with 32 filters and ReLU activation
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x) # Add a max pooling layer

    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(128, kernel_size=3, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)


    # Flattening Layer
    x = Flatten()(x)  # Flatten the output of the convolutional layers

    shared = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(x)
    shared = Dropout(0.5)(shared)

    # Count-specific hidden layers
    count_branch = Dense(32, activation='relu')(shared)
    count_branch = Dropout(0.2)(count_branch)

    count_output = Dense(
        num_classes,
        activation='softmax',
        name='count_output'
    )(count_branch)

    duration_branch = Dense(32, activation='relu')(shared)
    duration_branch = Dropout(0.2)(duration_branch) # try increasing drop out for more regularization? (increase if overfitting)

    duration_output = Dense(
        1,
        activation='linear',
        name='duration_output'
    )(duration_branch)


    model = Model(inputs=inputs, outputs=[count_output, duration_output])

    return model  # Return the compiled model

## Helper Functions

In [32]:
# function for running k-folds cross validation for any model 
def run_kfold_training(
    model_func, # function for selected model 
    X, # training split data 
    y, # training split labels 
    input_shape,
    num_classes, 
    feature_num,
    compile_kwargs,
    fit_kwargs=None,
    n_splits=5,
    random_state=42,
    shuffle=True,
    model_type=1,  # 1: single head model, 2: two head count and duration
):
    """
    Returns:
        dict: {"models", "histories", "cv_scores"}
    """

    if fit_kwargs is None:
        fit_kwargs = {}
    fit_kwargs = fit_kwargs.copy()


    kf = KFold(n_splits=n_splits, random_state=random_state, shuffle=shuffle)
    models = []
    histories = []
    cv_scores = []

    fold = 1
    for train_idx, val_idx in kf.split(X):
        print(f"Fold: {fold} {'='*65}")
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = model_func(input_shape, num_classes, feature_num)
        model.compile(**compile_kwargs)

        # single head models  
        if model_type ==1:
            history = model.fit(
                    X_train,
                    y_train,
                    validation_data=(X_val, y_val), epochs=epoch_num,)
            
            scores = model.evaluate(X_val, y_val, return_dict=True)

        
        # two head model
        elif model_type ==2:
            history = model.fit(
                    X_train,
                    {"count_output": y_train[:, 0],
                    "duration_output": y_train[:, 1],},
                    validation_data=(X_val,  
                        { "count_output": y_val[:, 0],
                        "duration_output": y_val[:, 1],})
                        ,epochs=epoch_num )
            
            scores = model.evaluate(X_val, 
                                    {"count_output": y_val[:, 0],
                                    "duration_output": y_val[:, 1],}, 
                                    return_dict=True)
        fold += 1
        
        models.append(model)
        histories.append(history)
        cv_scores.append(scores)

    return {
        "models": models,
        "histories": histories,
        "cv_scores": cv_scores,
    }

## Training 

### Single Head Count

In [33]:
# define variables and metrics used for training 
compile_kwargs_contr ={
        "optimizer": "adam",
        "loss": "sparse_categorical_crossentropy",
        "metrics": ['accuracy']
    }

num_classes = len(np.unique(y_contractions))    
feature_num = 1 
model = CNN_model_contraction(input_shape, num_classes, 1)

optimizer=compile_kwargs_contr["optimizer"]
loss=compile_kwargs_contr["loss"]
metrics=compile_kwargs_contr["metrics"]

/Users/zeynepozkaya/anaconda3/envs/tf_gpu/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
# k-folds cross validation 
results_single_head_contraction = run_kfold_training(
    model_func=CNN_model_contraction,
    X=X_train_full,
    y=y_train_full_contr,
    input_shape=input_shape,
    num_classes=num_classes,
    feature_num=feature_num,
    compile_kwargs=compile_kwargs_contr,
    fit_kwargs={"epochs": 10},
    n_splits=5)

Fold: 1 =================================================================
Epoch 1/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 17s 53ms/step - accuracy: 0.8147 - loss: 1.1148 - val_accuracy: 0.8560 - val_loss: 0.5985
Epoch 2/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 15s 51ms/step - accuracy: 0.8665 - loss: 0.5434 - val_accuracy: 0.8464 - val_loss: 0.6047
Epoch 3/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 13s 45ms/step - accuracy: 0.8905 - loss: 0.4212 - val_accuracy: 0.8626 - val_loss: 0.6457
Epoch 4/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 13s 45ms/step - accuracy: 0.9062 - loss: 0.3228 - val_accuracy: 0.8573 - val_loss: 0.6470
Epoch 5/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 13s 45ms/step - accuracy: 0.9288 - loss: 0.2403 - val_accuracy: 0.8613 - val_loss: 0.8103
Epoch 6/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 13s 45ms/step - accuracy: 0.9410 - loss: 0.1881 - val_accuracy: 0.8547 - val_loss: 0.8535
Epoch 7/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 13s 45ms/step - accuracy: 0.9587 - loss: 0.1419 - val_accuracy: 0.8582 - val_loss: 1.1013
Epoch 8/10
286/2

In [34]:
# fit model 
model = CNN_model_contraction(input_shape, num_classes, feature_num)
model.compile(optimizer=optimizer,loss=loss, metrics=compile_kwargs_contr["metrics"] )
model_history_contraction = model.fit( X_train_full,y_train_full_contr,  epochs=10)
model.save("contraction.keras")  # saves architecture + weights + optimizer state

Epoch 1/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 20s 52ms/step - accuracy: 0.8194 - loss: 1.0515
Epoch 2/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 17s 47ms/step - accuracy: 0.8651 - loss: 0.5612
Epoch 3/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 19s 52ms/step - accuracy: 0.8815 - loss: 0.4495
Epoch 4/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 17s 48ms/step - accuracy: 0.9015 - loss: 0.3536
Epoch 5/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 20s 55ms/step - accuracy: 0.9214 - loss: 0.2672
Epoch 6/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 21s 60ms/step - accuracy: 0.9446 - loss: 0.1774
Epoch 7/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 17s 47ms/step - accuracy: 0.9508 - loss: 0.1595
Epoch 8/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 18s 50ms/step - accuracy: 0.9659 - loss: 0.1244
Epoch 9/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 19s 52ms/step - accuracy: 0.9688 - loss: 0.1097
Epoch 10/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 18s 50ms/step - accuracy: 0.9782 - loss: 0.0785


In [12]:
# cross validation results/model results 
cvScores_contraction = results_single_head_contraction["cv_scores"]
accuracies = [fold['accuracy'] for fold in cvScores_contraction]

avgScores = np.mean(accuracies)
stdScores = np.std(accuracies)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")

# full training results (test data not seen during cross val)
y_pred_train = model.predict(X_train_full)  
y_pred_train = np.argmax(y_pred_train, axis=1)   

# Predict on test data
y_pred_test = model.predict(X_test)   
y_pred_test = np.argmax(y_pred_test, axis=1)   


# Calculate accuracy
accuracy_training_contr = accuracy_score(y_train_full_contr, y_pred_train)   
accuracy_test_contr = accuracy_score(y_test_contr, y_pred_test)  

# Calculate F1 score
f1_training_contr = f1_score(y_train_full_contr, y_pred_train, average='weighted')  
f1_test_contr = f1_score(y_test_contr, y_pred_test, average='weighted')  

# Print accuracy and F1 score
print("Model scores---------------")
print("Training Accuracy :", accuracy_training_contr)  
print("Test Accuracy :", accuracy_test_contr)  
print("Training F1 Score :", f1_training_contr)   
print("Test F1 Score :", f1_test_contr) 

Average KFold Cross Validation Score: 0.8625701069831848
Standard Deviation KFold Cross Validation Score: 0.004735654714943808
357/357 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
Model scores---------------
Training Accuracy : 0.9955357142857143
Test Accuracy : 0.8686974789915967
Training F1 Score : 0.9954063713925312
Test F1 Score : 0.847403918133005


### Single Head Duration

In [13]:
# define variables and metrics used for training 
compile_kwargs_dur={
        "optimizer": "adam",
        "loss": "mae",
        "metrics": ["mae"]}

num_classes = len(np.unique(y_durations))    

model = CNN_model_contraction(input_shape, num_classes, feature_num)

optimizer=compile_kwargs_dur["optimizer"]
loss=compile_kwargs_dur["loss"]
metrics=compile_kwargs_dur["metrics"]

/Users/zeynepozkaya/anaconda3/envs/tf_gpu/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
# call functionf or training 
results_single_head_dur = run_kfold_training(
    model_func=CNN_model_duration,
    X=X_train_full,
    y=y_train_full_dur,
    input_shape=input_shape,
    num_classes=num_classes,
    feature_num=feature_num,
    compile_kwargs=compile_kwargs_dur,
    fit_kwargs={"epochs": 10},
    n_splits=5)

Fold: 1 =================================================================
Epoch 1/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 17s 54ms/step - loss: 0.4883 - mae: 0.4883 - val_loss: 0.3125 - val_mae: 0.3125
Epoch 2/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 15s 53ms/step - loss: 0.3216 - mae: 0.3216 - val_loss: 0.2926 - val_mae: 0.2926
Epoch 3/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 15s 52ms/step - loss: 0.3109 - mae: 0.3109 - val_loss: 0.2892 - val_mae: 0.2892
Epoch 4/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 15s 52ms/step - loss: 0.3004 - mae: 0.3004 - val_loss: 0.2912 - val_mae: 0.2912
Epoch 5/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 15s 52ms/step - loss: 0.2985 - mae: 0.2985 - val_loss: 0.2938 - val_mae: 0.2938
Epoch 6/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 15s 53ms/step - loss: 0.2893 - mae: 0.2893 - val_loss: 0.2891 - val_mae: 0.2891
Epoch 7/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 15s 52ms/step - loss: 0.2811 - mae: 0.2811 - val_loss: 0.2819 - val_mae: 0.2819
Epoch 8/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 19s 67ms/step - loss: 0.2738 - mae: 0.2738 - v

In [15]:
# fit model 
model_duration = CNN_model_duration(input_shape, num_classes, 1)
model_duration.compile(optimizer=optimizer,loss=loss, metrics=metrics)
model_history_contraction = model_duration.fit( X_train_full,y_train_full_dur,  epochs=10)
#model_duration.save("duration.keras")  # saves architecture + weights + optimizer state

Epoch 1/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 20s 53ms/step - loss: 0.4101 - mae: 0.4101
Epoch 2/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 19s 52ms/step - loss: 0.3076 - mae: 0.3076
Epoch 3/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 26s 72ms/step - loss: 0.2996 - mae: 0.2996
Epoch 4/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 23s 65ms/step - loss: 0.2976 - mae: 0.2976
Epoch 5/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 19s 55ms/step - loss: 0.2849 - mae: 0.2849
Epoch 6/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 18s 49ms/step - loss: 0.2847 - mae: 0.2847
Epoch 7/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 17s 49ms/step - loss: 0.2795 - mae: 0.2795
Epoch 8/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 17s 48ms/step - loss: 0.2753 - mae: 0.2753
Epoch 9/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 17s 48ms/step - loss: 0.2757 - mae: 0.2757
Epoch 10/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 17s 48ms/step - loss: 0.2669 - mae: 0.2669


In [16]:
# cross validation/model results
cvScores_duration = np.array(results_single_head_dur["cv_scores"])
mae = [fold['mae'] for fold in cvScores_duration]

avgScores = np.mean(mae)
stdScores = np.std(mae)

print(f"Average MAE KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation of MAE KFold Cross Validation Score: {stdScores}")
 
baseline = np.mean(y_train_full_dur)
mae_baseline = np.mean(np.abs(y_train_full_dur - baseline))

# full training results (test data not seen during cross val)
y_pred_train_dur = model_duration.predict(X_train_full) 

# Predict on test data
y_pred_test_dur = model_duration.predict(X_test) 
  
mae_training_dur = mean_absolute_error(y_train_full_dur, y_pred_train_dur)   
mae_test_dur= mean_absolute_error(y_test_dur, y_pred_test_dur)  
  
# Print accuracy and F1 score
print("Baseline MAE:", mae_baseline)
print("Training MAE :", mae_training_dur)
print("Test MAE :", mae_test_dur)

Average MAE KFold Cross Validation Score: 0.29884097576141355
Standard Deviation of MAE KFold Cross Validation Score: 0.007255643612426481
357/357 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step
Baseline MAE: 0.5596337799825811
Training MAE : 0.24738429486751556
Test MAE : 0.2936214506626129


### Two Head 

In [17]:
# define variables and metrics used for training 

compile_kwargs_2h = {
        "optimizer": "adam",
        "loss": {
            "count_output": "sparse_categorical_crossentropy",
            "duration_output": "mae",
        },
        "metrics": {
            "count_output": ["accuracy"],
            "duration_output": ["mae"],
        },
    }

num_classes = len(np.unique(y))    
model_2h = CNN_model_twohead(input_shape, num_classes, 1)

optimizer=compile_kwargs_dur["optimizer"]
loss=compile_kwargs_2h["loss"]
metrics=compile_kwargs_2h["metrics"]

/Users/zeynepozkaya/anaconda3/envs/tf_gpu/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [18]:
# call function for training 
results_twohead = run_kfold_training(
    model_func=CNN_model_twohead,
    X=X_train_full,
    y=y_train_full,
    input_shape=input_shape,
    num_classes=num_classes,
    feature_num=feature_num,
    compile_kwargs=compile_kwargs_2h,
    fit_kwargs={"epochs": 10},
    n_splits=5,
    model_type=2
)

cvScores = results_twohead["cv_scores"]
cvScores_contr = [score["count_output_accuracy"] * 100 for score in cvScores]
cvScores_dur = [score["duration_output_mae"] for score in cvScores]

Fold: 1 =================================================================
Epoch 1/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 37s 115ms/step - count_output_accuracy: 0.7295 - count_output_loss: 1.2468 - duration_output_loss: 0.9829 - duration_output_mae: 0.9834 - loss: 2.7259 - val_count_output_accuracy: 0.7807 - val_count_output_loss: 1.4159 - val_duration_output_loss: 1.3402 - val_duration_output_mae: 1.3389 - val_loss: 3.2006
Epoch 2/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 32s 112ms/step - count_output_accuracy: 0.7401 - count_output_loss: 0.8435 - duration_output_loss: 0.4566 - duration_output_mae: 0.4568 - loss: 1.7450 - val_count_output_accuracy: 0.7908 - val_count_output_loss: 0.8045 - val_duration_output_loss: 0.3709 - val_duration_output_mae: 0.3726 - val_loss: 1.5668
Epoch 3/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 32s 111ms/step - count_output_accuracy: 0.7397 - count_output_loss: 0.6955 - duration_output_loss: 0.3976 - duration_output_mae: 0.3977 - loss: 1.4735 - val_count_output_accuracy: 0.7641 - val

In [19]:
# fit model 
model_twohead = CNN_model_twohead(input_shape,num_classes,1)
model_twohead.compile(optimizer=optimizer,loss=loss, metrics=metrics)
model_history_twohead = model_twohead.fit(
    X_train_full,
    {"count_output": y_train_full[:, 0], "duration_output": y_train_full[:, 1]},
    epochs=10) 
#model_duration.save("duration.keras")  # saves architecture + weights + optimizer state

Epoch 1/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 44s 111ms/step - count_output_accuracy: 0.7272 - count_output_loss: 1.2227 - duration_output_loss: 0.9257 - duration_output_mae: 0.9257 - loss: 2.6704
Epoch 2/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 40s 112ms/step - count_output_accuracy: 0.7495 - count_output_loss: 0.7890 - duration_output_loss: 0.4297 - duration_output_mae: 0.4297 - loss: 1.6956
Epoch 3/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 41s 114ms/step - count_output_accuracy: 0.7542 - count_output_loss: 0.7184 - duration_output_loss: 0.3794 - duration_output_mae: 0.3794 - loss: 1.5151
Epoch 4/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 40s 112ms/step - count_output_accuracy: 0.7784 - count_output_loss: 0.6658 - duration_output_loss: 0.3597 - duration_output_mae: 0.3597 - loss: 1.3683
Epoch 5/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 40s 112ms/step - count_output_accuracy: 0.7924 - count_output_loss: 0.6348 - duration_output_loss: 0.3422 - duration_output_mae: 0.3422 - loss: 1.3048
Epoch 6/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 40s 1

In [20]:
# cross validation/model resutls 
cvScores = results_twohead["cv_scores"]
cvScores_contr = [score["count_output_accuracy"] * 100 for score in cvScores]
cvScores_dur = [score["duration_output_mae"] for score in cvScores]


# cross validation results 
avgScores_contr = np.mean(cvScores_contr)
stdScores_contr = np.std(cvScores_contr)

avgScores_dur = np.mean(cvScores_dur)
stdScores_dur = np.std(cvScores_dur)

print(f"Average KFold Cross Validation Score for contraction: {avgScores_contr}")
print(f"Standard Deviation KFold Cross Validation Score for contractionon: {stdScores_contr}")

print(f"Average KFold Cross Validation Score for duration: {avgScores_dur}")
print(f"Standard Deviation KFold Cross Validation Score for duration: {stdScores_dur}")

y_pred_train = model_twohead.predict(X_train_full) 
y_pred_test = model_twohead.predict(X_test) 

y_pred_train_contraction = y_pred_train[0]
y_pred_train_contraction = np.argmax(y_pred_train_contraction, axis=1)   

y_pred_train_dur = y_pred_train[1]

y_pred_test_contraction = y_pred_test[0]
y_pred_test_contraction = np.argmax(y_pred_test_contraction, axis=1)   

y_pred_test_dur = y_pred_test[1]

# Calculate accuracy
accuracy_training_contraction = accuracy_score(y_train_full[:,0], y_pred_train_contraction)   
accuracy_test_contraction = accuracy_score(y_test[:,0], y_pred_test_contraction)  

# Calculate F1 score
f1_training_contraction = f1_score(y_train_full[:,0], y_pred_train_contraction, average='weighted')  
f1_test_contraction = f1_score(y_test[:,0], y_pred_test_contraction, average='weighted')  

# MAE 
baseline = np.mean(y_train_full[:,1])
mae_baseline = np.mean(np.abs(y_train_full[:,1] - baseline))
mae_training_dur = mean_absolute_error(y_train_full[:,1], y_pred_train_dur)   
mae_test_dur= mean_absolute_error(y_test[:,1], y_pred_test_dur)  
  
# Print accuracy and F1 score
print("Training Accuracy :", accuracy_training_contraction)  
print("Test Accuracy :", accuracy_test_contraction)  
print("Training F1 Score :", f1_training_contraction)   
print("Test F1 Score :", f1_test_contraction)   
print("----------------------") 
print("Baseline MAE:", mae_baseline)
print("Training MAE :", mae_training_dur)
print("Test MAE :", mae_test_dur)


Average KFold Cross Validation Score for contraction: 85.57420134544373
Standard Deviation KFold Cross Validation Score for contractionon: 1.0524060138730167
Average KFold Cross Validation Score for duration: 0.30392932891845703
Standard Deviation KFold Cross Validation Score for duration: 0.027326570408307684
357/357 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step
Training Accuracy : 0.8834033613445378
Test Accuracy : 0.8795518207282913
Training F1 Score : 0.8586099085867841
Test F1 Score : 0.8555236186047417
----------------------
Baseline MAE: 0.5596337799825811
Training MAE : 0.2892066240310669
Test MAE : 0.2959251403808594
